# Train real matched controls and load reference baselines

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='03_run_baselines_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='03_run_baselines')
print('Experiment worker:', runtime.python)


Every trained control starts again from the original pinned checkpoint. The released Jeva model uses its official runtime; the AR reference uses Qwen2.5-3B-Instruct. Missing assets cause an explicit error. API calls remain disabled unless enable_paid_api is deliberately enabled in the shared configuration.

In [ ]:
run_python(r"""
for backbone in cfg['backbones']:
    methods = ['serialized_dlm'] + (['ar'] if backbone.startswith('nemotron') else [])
    for method in methods:
        print(w.train_run(backbone, method=method, seed=SEED, device=DEVICE))
        gc.collect()
""")

In [ ]:
run_python(r"""
for name in cfg['references'] + ['bge']:
    runner = w.baseline_runner(name, device=DEVICE)
    print(name, type(runner).__name__)
    del runner
    gc.collect()
""")